# In-Memory Vector Store

**`InMemoryVectorStore`** is the simplest vector store in LangChain. It ships with **`langchain-core`**, so there is nothing extra to install, and it keeps every chunk in a **plain Python dictionary** in your notebook's memory.

- **Nothing to install or run:** no server, no C++ library, no database files.
- **Easy to look inside:** the whole store is one dict, `vectorstore.store`, that you can print.
- **Exact search:** every question is compared with **every** stored vector using cosine similarity, so the results are always exact.
- **Same LangChain interface:** `from_documents`, `similarity_search`, `as_retriever`, ... work just like with Chroma and FAISS, so you can swap it in or out by changing one line.

The catch: everything lives in RAM and **disappears when the kernel restarts**, unless you save it with `dump()`. And because it compares the question with every vector, search gets slower as the collection grows.

> **In one sentence:** `InMemoryVectorStore` is a Python dict of vectors with exact cosine search, ideal for learning, tests and small prototypes.

This notebook uses `InMemoryVectorStore` from `langchain_core.vectorstores` (langchain-core 1.6.x).

## InMemoryVectorStore vs. FAISS vs. Chroma

| | InMemoryVectorStore | FAISS | Chroma |
|---|---|---|---|
| **What it is** | A Python dict inside LangChain | A vector search **library** | A vector **database** |
| **Install** | Nothing: part of `langchain-core` | `faiss-cpu` | `chromadb` + `langchain-chroma` |
| **Search** | Exact: compares the question with every vector | Exact (Flat) or approximate (IVF, HNSW, PQ) | Approximate (HNSW) |
| **Score from `similarity_search_with_score`** | Cosine **similarity**: **higher = better** | Squared L2 **distance**: lower = better | Squared L2 **distance**: lower = better (by default) |
| **Saving to disk** | Manual: `dump()` / `load()`, one JSON file | Manual: `save_local()` / `load_local()`, a pickle | Automatic with `persist_directory` |
| **Metadata filtering** | A Python function, applied **before** the search | A dict or function, applied **after** the search | `where` / `where_document`, inside the database |
| **Adding an existing id** | Overwrites it (upsert) | Raises `ValueError` | Overwrites it (upsert) |
| **Scales to** | Small collections: a few thousand chunks | Millions to billions of vectors | Millions of vectors |
| **Best for** | Learning, unit tests, demos, small prototypes | Fast local search over large collections that rarely change | Apps that need CRUD, filtering and persistence |

All three plug into LangChain the same way (`from_documents`, `similarity_search`, `as_retriever`), so the retrievers and LCEL chains from the previous notebooks work unchanged.

## How it works

There is no index, id mapping or docstore like in FAISS. Each chunk's vector, text and metadata sit together in **one dict entry**:

```
InMemoryVectorStore
 │
 ├── embedding    OpenAIEmbeddings(...): embeds the chunks when you add them, and the question when you search
 │
 └── store        a plain Python dict, one entry per chunk:
                  {
                    "chunk_0": {"id": "chunk_0",
                                "vector": [-0.029, -0.008, 0.022, ...],     ← 1536 numbers
                                "text": "AI Agents: An Introduction ...",
                                "metadata": {"source": "data/raw/text/ai_agents.txt", "topic": "agents"}},
                    "chunk_1": {...},
                    ...
                  }
```

What happens in `similarity_search(query, k=3)`:

```
"What is reinforcement learning?"
        │
        ▼
 1. Embed the question          embeddings.embed_query(query)
        │
        ▼
 2. Filter (optional)           keep only the chunks where filter(doc) returns True
        │
        ▼
 3. Compare with every chunk    cosine similarity between the question vector and each stored vector
        │
        ▼
 4. Keep the top k              sort by similarity, highest first, keep k = 3
        │
        ▼
 5. Return Documents            id + text + metadata of each chunk
```

There is nothing to build or train: adding a chunk just embeds it and puts it in the dict.

## Similarity scores

`similarity_search_with_score` returns the **cosine similarity** between the question and each chunk:

| Score | Meaning |
|---|---|
| `1.0` | Same direction: identical meaning |
| `0` | Unrelated |
| `-1.0` | Opposite |

So **higher = better**. This is the **opposite** of FAISS and Chroma, which return **distances** where lower = better. Watch out when you reuse a score threshold from another notebook.

For normalised embeddings such as OpenAI's, the two are directly linked:

$$\text{cosine similarity} = 1 - \frac{\text{squared L2 distance}}{2}$$

For example, the FAISS distance of `0.65` for *"What is reinforcement learning?"* in the previous notebook is a cosine similarity of about `0.68` here. The ranking is the same, only the numbers differ.

**No relevance scores.** `similarity_search_with_relevance_scores` and `as_retriever(search_type="similarity_score_threshold")` raise `NotImplementedError`, because `InMemoryVectorStore` doesn't define how to turn its scores into relevance scores. Use `similarity_search_with_score` and keep the good matches yourself:

```python
results = vectorstore.similarity_search_with_score(query, k=5)
good_matches = [(doc, score) for doc, score in results if score >= 0.5]
```

## Working with an InMemoryVectorStore in LangChain

| Method | What it does |
|---|---|
| `InMemoryVectorStore(embedding)` | Create an empty store |
| `InMemoryVectorStore.from_documents(docs, embedding, ids=...)` | Embed the chunks and build a new store |
| `vectorstore.add_documents(docs, ids=...)` | Embed and add more chunks. An id that already exists is **overwritten** |
| `vectorstore.similarity_search(query, k=4, filter=...)` | Return the `k` most similar `Document`s (default `k=4`) |
| `vectorstore.similarity_search_with_score(query, k=4)` | Return `(Document, cosine similarity)` pairs |
| `vectorstore.max_marginal_relevance_search(query, k, fetch_k)` | Return relevant **and** varied chunks (MMR) |
| `vectorstore.get_by_ids([...])` | Fetch chunks by id, without searching |
| `vectorstore.delete(ids=[...])` | Remove chunks by id |
| `vectorstore.store` | The underlying dict: print it to look inside |
| `vectorstore.dump(path)` | Save the whole store to a JSON file |
| `InMemoryVectorStore.load(path, embedding)` | Load a saved store |
| `vectorstore.as_retriever(search_kwargs={"k": 3})` | Use it in a RAG chain, like the Chroma and FAISS retrievers |

Every method also has an async version: `aadd_documents`, `asimilarity_search`, `adelete`, ...

### Build, save and load

```python
from langchain_core.vectorstores import InMemoryVectorStore

# Build the store from the chunks
vectorstore = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)

# Save it to a JSON file
vectorstore.dump("vectorstore/inmemory_store.json")

# Later, or in another notebook: load it again without re-embedding
vectorstore = InMemoryVectorStore.load(
    "vectorstore/inmemory_store.json",
    embeddings,            # the same embedding model used to build it
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
```

### Things to watch out for

- **Gone on restart.** The store lives in the kernel's memory. Restarting the kernel or closing the notebook deletes it, unless you called `dump()`. Changes made after the last `dump()` are lost too.
- **`dump()` writes JSON, not a pickle.** Unlike FAISS's `index.pkl`, loading it can't run code, so there is no `allow_dangerous_deserialization` flag. But the file stores every vector as text (1536 numbers per chunk with `text-embedding-3-small`), so it is large.
- **Adding an existing id overwrites it.** `add_documents` with an id that is already in the store replaces that chunk, with no error. Without ids, LangChain creates random ids, so running the same cell twice adds **duplicate** chunks.
- **Same embedding model.** As with Chroma and FAISS, always load and query with the model that built the store.

## Metadata filtering

`filter` takes a **Python function**. It gets each chunk as a `Document` and returns `True` to keep it:

```python
# One topic
results = vectorstore.similarity_search(
    "What is reinforcement learning?",
    k=3,
    filter=lambda doc: doc.metadata["topic"] == "ml",
)

# ml or dl
results = vectorstore.similarity_search(
    "What is reinforcement learning?",
    k=3,
    filter=lambda doc: doc.metadata["topic"] in ["ml", "dl"],
)

# The same filter in a retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "filter": lambda doc: doc.metadata["topic"] == "dl"}
)
```

Two differences from FAISS:

- **Functions only, no dict filters.** `filter={"topic": "ml"}` raises `TypeError: 'dict' object is not callable`. In exchange, the function can check anything: the metadata, the text (`"reward" in doc.page_content`), or both.
- **Filtering happens before the search.** The non-matching chunks are dropped first, then the rest are ranked. You always get `k` results as long as at least `k` chunks match, and there is no `fetch_k` to tune.

```
FAISS:                 20 nearest chunks  ─►  keep the matching ones  ─►  first k    (can return fewer than k)

InMemoryVectorStore:   keep the matching chunks  ─►  rank all of them  ─►  top k     (always k, if k chunks match)
```

## Strengths and limits

**Strengths**

- Nothing to install or run: it is part of `langchain-core`
- Exact search, with no approximate index that could miss a close match
- Easy to inspect and debug: the whole store is one Python dict
- Upsert by id, delete by id, and `get_by_ids`
- Filters can be any Python function, and they are applied before the search
- Saves to a readable JSON file that is safe to load

**Limits to keep in mind**

- **Everything in RAM, nothing saved automatically.** The store is lost when the kernel stops, unless you `dump()` it.
- **Slow on large collections.** Every search compares the question with every stored vector, so search time grows with the number of chunks. The vectors are kept as Python lists, which also uses much more memory than FAISS's compact arrays.
- **No relevance scores.** `similarity_search_with_relevance_scores` and the `similarity_score_threshold` retriever raise `NotImplementedError`.
- **One process only.** There is no server, so other programs or users can't share the store.
- **Large save files.** The JSON file stores every vector as text.

## Key takeaways

- **`InMemoryVectorStore`** ships with **langchain-core** and keeps every chunk (id, vector, text and metadata) in one **Python dict**.
- Search is **exact**: the question is compared with **every** vector using **cosine similarity**.
- Scores are **similarities**: **higher = better**, unlike the FAISS and Chroma distances. Relevance scores are not supported.
- **Nothing is saved automatically**: `dump()` writes a JSON file and `load()` reads it back.
- `add_documents` with an existing id **overwrites** that chunk, so fixed ids prevent duplicates.
- Filters are **Python functions** applied **before** the search, so you get `k` results whenever at least `k` chunks match.
- `as_retriever()` plugs it into the same LCEL and conversational RAG chains as Chroma and FAISS. Use it to learn and prototype, then switch to FAISS or Chroma when the collection grows or must persist.

**Next:** we'll build an `InMemoryVectorStore` from our text chunks, run similarity searches with and without filters, and save and load it.

In [1]:
## Setup: move to the project root and load the API key from .env
import os
from pathlib import Path
from dotenv import load_dotenv

cwd = Path.cwd().resolve()
project_root = next(p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists())
os.chdir(project_root)

load_dotenv()
print(f"Working directory set to: {project_root}")

Working directory set to: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [2]:
## Load libraries
import warnings
warnings.filterwarnings("ignore")

# LangChain core imports
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.vectorstores import InMemoryVectorStore

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader

## Data Ingestion

In [3]:
## Load the text files
doc_dir = "./data/raw/text"
loader = DirectoryLoader(
    doc_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  {doc.metadata['source']}: {len(doc.page_content)} characters")

Loaded 4 documents
  data/raw/text/ai_agents.txt: 4117 characters
  data/raw/text/machine_learning_algorithms.txt: 4220 characters
  data/raw/text/reinforcement_learning.txt: 3909 characters
  data/raw/text/deep_learning.txt: 2505 characters


In [4]:
## Split the documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],   # try paragraphs first, then lines, then words, then characters
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")
print(f"Metadata: {chunks[0].metadata}")

Created 44 chunks
Metadata: {'source': 'data/raw/text/ai_agents.txt'}


## Create the In-Memory Vector Store

In [5]:
## Embed the chunks and keep them in memory
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
)

print(f"Stored {len(vectorstore.store)} chunks in memory")

Stored 44 chunks in memory


## Build the RAG Chain

In [6]:
## Retriever, LLM and prompt
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

llm = ChatOpenAI(model="gpt-5", temperature=0)

custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.
Mention the source file(s) you used.

Context:
{context}

Question: {question}

Answer:""")

In [7]:
## Format the retrieved chunks for the prompt, with a [Source: ...] label on each
def format_docs(docs):
    return "\n\n".join(f"[Source: {doc.metadata['source']}]\n{doc.page_content}" for doc in docs)

In [8]:
## Build the chain using LCEL
rag_chain_lcel = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['InMemoryVectorStore', 'OpenAIEmbeddings'], vectorstore=<langchain_core.vectorstores.in_memory.InMemoryVectorStore object at 0x12b748590>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question.\nIf you don't know the answer based on the context, say you don't know.\nMention the source file(s) you used.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.1', 'langchain-openai': '1.6.2'}}, profile={'name': 'GPT-5', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False

In [9]:
## Helper: ask the RAG chain a question and show the retrieved chunks with their similarity scores
def ask_rag(question, show_sources=True):
    """Run rag_chain_lcel on a question, print the answer and (optionally) the chunks it was given."""
    answer = rag_chain_lcel.invoke(question)

    print(f"Question: {question}\n")
    print(f"Answer:\n{answer}")

    if show_sources:
        # The same search the retriever ran, but with scores: cosine similarity, higher = better
        results = vectorstore.similarity_search_with_score(question, k=retriever.search_kwargs["k"])
        print(f"\nRetrieved {len(results)} chunks:")
        for i, (doc, score) in enumerate(results, start=1):
            print(f"\n --- Chunk {i} --- id: {doc.id} | similarity: {score:.4f} | source: {doc.metadata['source']}")
            print(doc.page_content[:200] + "...")

    return answer


answer = ask_rag("What is reinforcement learning?")

Question: What is reinforcement learning?

Answer:
Reinforcement learning is a branch of machine learning where an agent learns through trial and error by interacting with an environment: it takes actions, receives rewards or penalties, and gradually learns a strategy (policy) that maximizes total reward over time.

Sources: data/raw/text/reinforcement_learning.txt; data/raw/text/machine_learning_algorithms.txt

Retrieved 3 chunks:

 --- Chunk 1 --- id: chunk_23 | similarity: 0.6758 | source: data/raw/text/reinforcement_learning.txt
Reinforcement Learning: An Introduction

Reinforcement learning (RL) is a branch of machine learning in which an agent learns to make decisions by interacting with an environment. Instead of learning ...

 --- Chunk 2 --- id: chunk_19 | similarity: 0.6572 | source: data/raw/text/machine_learning_algorithms.txt
Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards or penalties

In [10]:
answer = ask_rag("What are the types of neural networks?")

Question: What are the types of neural networks?

Answer:
- Convolutional Neural Networks (CNNs)
- Recurrent Neural Networks (RNNs)
- Long Short-Term Memory (LSTM) networks

Source: data/raw/text/deep_learning.txt

Retrieved 3 chunks:

 --- Chunk 1 --- id: chunk_37 | similarity: 0.5696 | source: data/raw/text/deep_learning.txt
A neural network is made up of neurons organized into layers: an input layer, one or more hidden layers, and an output layer. Each connection between neurons has a weight. During training, the network...

 --- Chunk 2 --- id: chunk_39 | similarity: 0.5059 | source: data/raw/text/deep_learning.txt
Common Architectures
Convolutional Neural Networks (CNNs) are designed for images. They use filters to detect edges, textures, and shapes, and are widely used in image classification and object detect...

 --- Chunk 3 --- id: chunk_35 | similarity: 0.4085 | source: data/raw/text/deep_learning.txt
Deep Learning: An Introduction

Deep learning is a subfield of machine lear